In [1]:
!pip install requests pandas numpy -q

In [2]:
!pip install ccxt -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.6/219.6 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.4/132.4 kB 11.9 MB/s eta 0:00:00


In [3]:
# 1. 라이브러리 설치 및 불러오기
!pip install -q finance-datareader tqdm

import datetime
import time
import FinanceDataReader as fdr
import pandas as pd
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

print("🎯 [속도 개선판] 종가배팅 매수 후보 종목 스크리닝을 시작합니다...\n")

# 2. 날짜 설정 (이평선 및 평균거래량 계산용)
start_date = (datetime.date.today() - datetime.timedelta(days=120)).strftime(
    "%Y-%m-%d"
)

# 3. 국내 시장 전체 종목 리스트 획득
df_krx = fdr.StockListing("KRX")
df_krx = df_krx[df_krx["Market"].isin(["KOSPI", "KOSDAQ"])]

# 4. 당일 거래대금 기준 상위 50개 우선 추출
df_krx["Amount"] = pd.to_numeric(df_krx["Amount"], errors="coerce")
df_krx = df_krx.dropna(subset=["Amount"])
top_50 = df_krx.nlargest(50, "Amount").copy()


# 5. 종목 1개를 검사하는 함수 (병렬 실행용으로 분리)
def check_stock(row, retries=1):
    code = row["Code"]
    name = row["Name"]

    for attempt in range(retries + 1):
        try:
            df_price = fdr.DataReader(code, start=start_date)
            if len(df_price) < 60:
                return None

            # 이동평균선 계산
            df_price["MA5"] = df_price["Close"].rolling(window=5).mean()
            df_price["MA20"] = df_price["Close"].rolling(window=20).mean()
            df_price["MA60"] = df_price["Close"].rolling(window=60).mean()

            # 최근 20거래일 평균 거래량 계산 (오늘 제외)
            df_price["Vol_MA20"] = (
                df_price["Volume"].shift(1).rolling(window=20).mean()
            )

            latest = df_price.iloc[-1]
            close = latest["Close"]
            volume = latest["Volume"]

            ma5 = latest["MA5"]
            ma20 = latest["MA20"]
            ma60 = latest["MA60"]
            vol_ma20 = latest["Vol_MA20"]

            # 조건 1: 단기 이평선 정배열
            if not (close > ma20 and close > ma60 and ma5 > ma20):
                return None

            # 조건 2: 거래량 300% 이상 폭발
            if volume < (vol_ma20 * 3.0):
                return None

            amount_idx = round(row["Amount"] / 100000000)
            return {
                "종목명": name,
                "현재가": f"{int(close):,}",
                "20일선": f"{int(ma20):,}",
                "거래대금": f"{amount_idx:,}억",
            }

        except Exception:
            if attempt < retries:
                time.sleep(1 + attempt)  # 재시도 전 잠깐 대기 (429 대응)
                continue
            return None


# 6. 병렬로 종목 검사 실행
results = []
MAX_WORKERS = 12  # 너무 높이면 429(요청 과다) 에러 위험 → 10~15 권장

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(check_stock, row): row for _, row in top_50.iterrows()}
    for future in tqdm(as_completed(futures), total=len(futures), desc="종목 조건 검증 중"):
        result = future.result()
        if result:
            results.append(result)

# 7. 결과 출력 (모바일 최적화 화면)
print("\n🔥 [종가배팅 필터링 결과] 🔥")
if results:
    final_df = pd.DataFrame(results)
    display(final_df.style.hide(axis="index"))
else:
    print("오늘 조건(거래대금 상위 + 이평 정배열 + 거래량 300% 폭발)을 만족하는 종목이 없습니다.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.0 MB/s eta 0:00:00
🎯 [속도 개선판] 종가배팅 매수 후보 종목 스크리닝을 시작합니다...



종목 조건 검증 중:   0%|          | 0/50 [00:00<?, ?it/s]


🔥 [종가배팅 필터링 결과] 🔥


종목명,현재가,20일선,거래대금
고려아연,"1,234,000","1,018,850","1,060억"
LG생활건강,"324,000","261,750",903억


In [4]:
# 1. 라이브러리 설치 및 불러오기
!pip install -q finance-datareader tqdm

import datetime
import FinanceDataReader as fdr
import pandas as pd
from tqdm.notebook import tqdm

print("🎯 [눌림목 지지] 종가배팅 매수 후보 종목 스크리닝을 시작합니다...\n")

# 2. 날짜 설정
start_date = (datetime.date.today() - datetime.timedelta(days=150)).strftime("%Y-%m-%d")

# 3. 국내 시장 전체 종목 리스트
df_krx = fdr.StockListing("KRX")
df_krx = df_krx[df_krx["Market"].isin(["KOSPI", "KOSDAQ"])]

# 4. 당일 거래대금 상위 80개 (눌림목은 후보 폭을 넓게)
df_krx["Amount"] = pd.to_numeric(df_krx["Amount"], errors="coerce")
df_krx = df_krx.dropna(subset=["Amount"])
top_list = df_krx.nlargest(80, "Amount").copy()

results = []

# 5. 눌림목 지지 조건 검증
for idx, row in tqdm(top_list.iterrows(), total=80, desc="눌림목 조건 검증 중"):
    code = row["Code"]
    name = row["Name"]

    try:
        df = fdr.DataReader(code, start=start_date)
        if len(df) < 65:
            continue

        # 이동평균선
        df["MA5"]  = df["Close"].rolling(5).mean()
        df["MA20"] = df["Close"].rolling(20).mean()
        df["MA60"] = df["Close"].rolling(60).mean()

        # 거래량 평균 (오늘 제외)
        df["Vol_MA20"] = df["Volume"].shift(1).rolling(20).mean()

        latest = df.iloc[-1]
        close, high, low, op = latest["Close"], latest["High"], latest["Low"], latest["Open"]
        volume = latest["Volume"]
        ma5, ma20, ma60, vol_ma20 = latest["MA5"], latest["MA20"], latest["MA60"], latest["Vol_MA20"]

        # 최근 5일 고점 (조정 폭 계산용)
        recent_high = df["High"].iloc[-15:].max()

        # ─── 🔍 눌림목 지지 핵심 조건 ───

        # 조건 1: 상승 추세 유지 (20일선 > 60일선, 60일선 우상향)
        ma60_5ago = df["MA60"].iloc[-6]
        if not (ma20 > ma60 and ma60 > ma60_5ago):
            continue

        # 조건 2: 20일선 위에서 마감 (추세 위 지지)
        if close < ma20:
            continue

        # 조건 3: 5일선 근처 또는 아래로 조정 (단기 과열 해소)
        #         → 종가가 5일선의 ±3% 이내 (5일선을 타고 내려온 눌림 구간)
        if not (close <= ma5 * 1.03):
            continue

        # 조건 4: 최근 고점 대비 적당한 조정 (3% ~ 15% 사이 눌림)
        pullback = (recent_high - close) / recent_high
        if not (0.03 <= pullback <= 0.15):
            continue

        # 조건 5: 거래량 감소 (매도 소진) — 평균의 120% 미만
        if volume > (vol_ma20 * 1.2):
            continue

        # 조건 6: 종가가 저가 위에서 지지 (장중 밀렸다 회복 = 지지 확인)
        #         → 종가가 당일 (저가~고가) 범위의 하위 40% 이상에 위치
        candle_range = high - low
        if candle_range > 0:
            close_pos = (close - low) / candle_range
            if close_pos < 0.4:
                continue

        # 통과 종목 저장
        amount_idx = round(row["Amount"] / 100000000)
        results.append({
            "종목명": name,
            "현재가": f"{int(close):,}",
            "5일선": f"{int(ma5):,}",
            "20일선": f"{int(ma20):,}",
            "고점대비조정": f"{pullback*100:.1f}%",
            "거래량비율": f"{volume/vol_ma20*100:.0f}%",
            "거래대금": f"{amount_idx:,}억",
        })

    except:
        continue

# 6. 결과 출력
print("\n🔥 [눌림목 지지 필터링 결과] 🔥")
if results:
    final_df = pd.DataFrame(results)
    display(final_df.style.hide(axis="index"))
else:
    print("오늘 눌림목 지지 조건(상승추세 + 5일선 눌림 + 거래량 감소 + 종가 지지)을 만족하는 종목이 없습니다.")

🎯 [눌림목 지지] 종가배팅 매수 후보 종목 스크리닝을 시작합니다...



눌림목 조건 검증 중:   0%|          | 0/80 [00:00<?, ?it/s]


🔥 [눌림목 지지 필터링 결과] 🔥


종목명,현재가,5일선,20일선,고점대비조정,거래량비율,거래대금
삼성바이오로직스,"1,514,000","1,477,800","1,438,250",3.9%,71%,730억
삼성화재,"648,000","630,600","632,150",7.6%,63%,713억


In [5]:
# 1. 라이브러리 설치 및 불러오기
!pip install -q finance-datareader tqdm

import datetime
import FinanceDataReader as fdr
import pandas as pd
from tqdm.notebook import tqdm

print("🎯 [전고점 돌파] 종가마감 매수 후보 종목 스크리닝을 시작합니다...\n")

# 2. 날짜 설정
start_date = (datetime.date.today() - datetime.timedelta(days=150)).strftime("%Y-%m-%d")

# 3. 국내 시장 전체 종목 리스트
df_krx = fdr.StockListing("KRX")
df_krx = df_krx[df_krx["Market"].isin(["KOSPI", "KOSDAQ"])]

# 4. 당일 거래대금 상위 80개
df_krx["Amount"] = pd.to_numeric(df_krx["Amount"], errors="coerce")
df_krx = df_krx.dropna(subset=["Amount"])
top_list = df_krx.nlargest(80, "Amount").copy()

# 돌파 기준 기간 (N일 신고가)
BREAKOUT_DAYS = 60

results = []

# 5. 전고점 돌파 조건 검증
for idx, row in tqdm(top_list.iterrows(), total=80, desc="돌파 조건 검증 중"):
    code = row["Code"]
    name = row["Name"]

    try:
        df = fdr.DataReader(code, start=start_date)
        if len(df) < BREAKOUT_DAYS + 5:
            continue

        # 이동평균선
        df["MA5"]  = df["Close"].rolling(5).mean()
        df["MA20"] = df["Close"].rolling(20).mean()
        df["MA60"] = df["Close"].rolling(60).mean()

        # 거래량 평균 (오늘 제외)
        df["Vol_MA20"] = df["Volume"].shift(1).rolling(20).mean()

        latest = df.iloc[-1]
        close, high, low, op = latest["Close"], latest["High"], latest["Low"], latest["Open"]
        volume = latest["Volume"]
        ma5, ma20, ma60, vol_ma20 = latest["MA5"], latest["MA20"], latest["MA60"], latest["Vol_MA20"]

        # 전고점: 오늘 제외, 직전 N일간의 최고 종가
        prev_high_close = df["Close"].iloc[-(BREAKOUT_DAYS+1):-1].max()
        # 직전 N일간 최고 고가 (장중 고점 기준 돌파 강도 참고용)
        prev_high_high = df["High"].iloc[-(BREAKOUT_DAYS+1):-1].max()

        # ─── 🔍 전고점 돌파 종가마감 핵심 조건 ───

        # 조건 1: 당일 종가가 직전 N일 최고 종가를 돌파 (신고가 종가마감)
        if not (close > prev_high_close):
            continue

        # 조건 2: 추세 위 마감 (종가 > 20일선, 20일선 > 60일선 = 상승 추세)
        if not (close > ma20 and ma20 > ma60):
            continue

        # 조건 3: 양봉 마감 (종가 > 시가)
        if not (close > op):
            continue

        # 조건 4: 윗꼬리 짧음 — 종가가 당일 (저가~고가) 범위 상위 30% 안
        #         → 장 마감까지 매수세 유지, 고점 부근에서 마감
        candle_range = high - low
        if candle_range > 0:
            close_pos = (close - low) / candle_range
            if close_pos < 0.7:
                continue

        # 조건 5: 거래량 동반 (평균의 150% 이상) — 돌파의 신뢰도
        if volume < (vol_ma20 * 1.5):
            continue

        # 돌파 강도 (전고점 대비 종가 상승률)
        breakout_pct = (close - prev_high_close) / prev_high_close

        amount_idx = round(row["Amount"] / 100000000)
        results.append({
            "종목명": name,
            "현재가": f"{int(close):,}",
            "전고점(종가)": f"{int(prev_high_close):,}",
            "돌파강도": f"+{breakout_pct*100:.1f}%",
            "종가위치": f"{close_pos*100:.0f}%",
            "거래량비율": f"{volume/vol_ma20*100:.0f}%",
            "거래대금": f"{amount_idx:,}억",
        })

    except:
        continue

# 6. 결과 출력
print(f"\n🔥 [전고점({BREAKOUT_DAYS}일) 돌파 종가마감 결과] 🔥")
if results:
    final_df = pd.DataFrame(results)
    display(final_df.style.hide(axis="index"))
else:
    print(f"오늘 {BREAKOUT_DAYS}일 신고가 종가돌파 + 윗꼬리 짧음 + 거래량 동반 조건을 만족하는 종목이 없습니다.")

🎯 [전고점 돌파] 종가마감 매수 후보 종목 스크리닝을 시작합니다...



돌파 조건 검증 중:   0%|          | 0/80 [00:00<?, ?it/s]


🔥 [전고점(60일) 돌파 종가마감 결과] 🔥


종목명,현재가,전고점(종가),돌파강도,종가위치,거래량비율,거래대금
LG생활건강,"324,000","292,500",+10.8%,88%,319%,903억


In [6]:
# 1. 라이브러리 설치 및 불러오기
!pip install -q finance-datareader tqdm

import datetime
import FinanceDataReader as fdr
import pandas as pd
from tqdm.notebook import tqdm

print("🎯 [첫 거래량 장대양봉] 종가강세 매수 후보 종목 스크리닝을 시작합니다...\n")

# 2. 날짜 설정
start_date = (datetime.date.today() - datetime.timedelta(days=180)).strftime("%Y-%m-%d")

# 3. 국내 시장 전체 종목 리스트
df_krx = fdr.StockListing("KRX")
df_krx = df_krx[df_krx["Market"].isin(["KOSPI", "KOSDAQ"])]

# 4. 당일 거래대금 상위 100개 (바닥 종목 포착 위해 넓게)
df_krx["Amount"] = pd.to_numeric(df_krx["Amount"], errors="coerce")
df_krx = df_krx.dropna(subset=["Amount"])
top_list = df_krx.nlargest(100, "Amount").copy()

# "처음" 판정 기간 (이 기간 내 거래량 폭발이 없었어야 함)
LOOKBACK_DAYS = 60

results = []

# 5. 첫 거래량 장대양봉 조건 검증
for idx, row in tqdm(top_list.iterrows(), total=100, desc="장대양봉 조건 검증 중"):
    code = row["Code"]
    name = row["Name"]

    try:
        df = fdr.DataReader(code, start=start_date)
        if len(df) < LOOKBACK_DAYS + 25:
            continue

        # 이동평균선
        df["MA5"]  = df["Close"].rolling(5).mean()
        df["MA20"] = df["Close"].rolling(20).mean()
        df["MA60"] = df["Close"].rolling(60).mean()

        # 거래량 평균 (오늘 제외)
        df["Vol_MA20"] = df["Volume"].shift(1).rolling(20).mean()

        latest = df.iloc[-1]
        close, high, low, op = latest["Close"], latest["High"], latest["Low"], latest["Open"]
        prev_close = df["Close"].iloc[-2]
        volume = latest["Volume"]
        ma5, ma20, ma60, vol_ma20 = latest["MA5"], latest["MA20"], latest["MA60"], latest["Vol_MA20"]

        # ─── 🔍 첫 거래량 장대양봉 핵심 조건 ───

        # 조건 1: 당일 거래량 폭발 (평균의 300% 이상)
        if volume < (vol_ma20 * 3.0):
            continue

        # 조건 2: "처음" 검증 — 직전 LOOKBACK_DAYS일간 이런 거래량 폭발이 없었어야 함
        #         (오늘 제외, 과거 구간에서 평균 대비 3배 이상 터진 날이 0일)
        past = df.iloc[-(LOOKBACK_DAYS+1):-1].copy()
        past_explosion = (past["Volume"] > past["Vol_MA20"] * 3.0).sum()
        if past_explosion > 0:
            continue

        # 조건 3: 장대양봉 — 당일 상승률 (종가/전일종가) 6% 이상
        day_return = (close - prev_close) / prev_close
        if day_return < 0.06:
            continue

        # 조건 4: 양봉 몸통이 큼 — (종가-시가)/시가 4% 이상
        body = (close - op) / op
        if body < 0.04:
            continue

        # 조건 5: 종가 강세 — 종가가 당일 (저가~고가) 범위 상위 25% 안 (윗꼬리 짧음)
        candle_range = high - low
        if candle_range > 0:
            close_pos = (close - low) / candle_range
            if close_pos < 0.75:
                continue
        else:
            continue

        # 조건 6: 바닥/횡보 탈출 — 종가가 60일선 위로 마감 (장기 침체 탈출 신호)
        if close < ma60:
            continue

        amount_idx = round(row["Amount"] / 100000000)
        results.append({
            "종목명": name,
            "현재가": f"{int(close):,}",
            "당일상승률": f"+{day_return*100:.1f}%",
            "양봉몸통": f"+{body*100:.1f}%",
            "종가위치": f"{close_pos*100:.0f}%",
            "거래량비율": f"{volume/vol_ma20*100:.0f}%",
            "거래대금": f"{amount_idx:,}억",
        })

    except:
        continue

# 6. 결과 출력
print(f"\n🔥 [첫 거래량({LOOKBACK_DAYS}일내) 장대양봉 종가강세 결과] 🔥")
if results:
    final_df = pd.DataFrame(results)
    display(final_df.style.hide(axis="index"))
else:
    print(f"오늘 첫 거래량 폭발 + 장대양봉 + 종가강세 + 60일선 돌파 조건을 만족하는 종목이 없습니다.")

🎯 [첫 거래량 장대양봉] 종가강세 매수 후보 종목 스크리닝을 시작합니다...



장대양봉 조건 검증 중:   0%|          | 0/100 [00:00<?, ?it/s]


🔥 [첫 거래량(60일내) 장대양봉 종가강세 결과] 🔥
오늘 첫 거래량 폭발 + 장대양봉 + 종가강세 + 60일선 돌파 조건을 만족하는 종목이 없습니다.


In [7]:
# 1. 라이브러리 설치 및 불러오기
!pip install -q finance-datareader

import datetime
import FinanceDataReader as fdr
import pandas as pd

print("🚦 [시장 레짐 필터] 오늘 종가배팅 적합도를 점검합니다...\n")

# 2. 날짜 설정
start_date = (datetime.date.today() - datetime.timedelta(days=200)).strftime("%Y-%m-%d")

# 3. 지수 데이터 로드 (코스피, 코스닥, 변동성지수)
def load_index(symbol):
    df = fdr.DataReader(symbol, start=start_date)
    df["MA5"]  = df["Close"].rolling(5).mean()
    df["MA20"] = df["Close"].rolling(20).mean()
    df["MA60"] = df["Close"].rolling(60).mean()
    return df

kospi  = load_index("KS11")   # 코스피
kosdaq = load_index("KQ11")   # 코스닥

# 변동성지수(VKOSPI)는 심볼이 환경마다 다를 수 있어 예외 처리
vkospi = None
for sym in ["VKOSPI", "KSVKOSPI"]:
    try:
        vkospi = fdr.DataReader(sym, start=start_date)
        if len(vkospi) > 20:
            break
    except:
        continue

# ─── 점수 채점 시스템 ───
score = 0
max_score = 0
reasons = []

def check(condition, weight, label_pass, label_fail):
    global score, max_score
    max_score += weight
    if condition:
        score += weight
        reasons.append(f"✅ {label_pass} (+{weight})")
    else:
        reasons.append(f"❌ {label_fail} (0)")

# 4. 지수별 추세 점검 함수
def trend_status(df, name, weight_trend, weight_slope):
    latest = df.iloc[-1]
    close, ma20, ma60 = latest["Close"], latest["MA20"], latest["MA60"]
    ma60_5ago = df["MA60"].iloc[-6]

    # 추세: 지수가 20일선·60일선 위
    check(
        close > ma20 and close > ma60,
        weight_trend,
        f"{name} 추세 양호 (지수>20·60일선)",
        f"{name} 추세 약함 (지수<이평선)",
    )
    # 기울기: 60일선 우상향
    check(
        ma60 > ma60_5ago,
        weight_slope,
        f"{name} 60일선 우상향",
        f"{name} 60일선 횡보/하락",
    )

# 코스피 추세 (가중치 높게 — 전체 시장 방향)
trend_status(kospi, "코스피", 25, 10)
# 코스닥 추세 (종가배팅 주무대 — 가중치 가장 높게)
trend_status(kosdaq, "코스닥", 30, 15)

# 5. 거래대금 추세 (코스닥 기준 — 최근 5일 평균 vs 직전 20일 평균)
kq_vol_recent = kosdaq["Volume"].iloc[-5:].mean()
kq_vol_base   = kosdaq["Volume"].iloc[-25:-5].mean()
check(
    kq_vol_recent > kq_vol_base,
    10,
    f"코스닥 거래량 증가 추세 (최근5일 {kq_vol_recent/kq_vol_base*100:.0f}%)",
    f"코스닥 거래량 위축 (최근5일 {kq_vol_recent/kq_vol_base*100:.0f}%)",
)

# 6. 변동성 점검 (VKOSPI) — 너무 높으면 휩쏘 위험으로 감점
if vkospi is not None and len(vkospi) > 20:
    vk_now = vkospi["Close"].iloc[-1]
    vk_ma20 = vkospi["Close"].iloc[-20:].mean()
    # 변동성이 20일 평균의 130% 미만이면 안정적
    check(
        vk_now < vk_ma20 * 1.3 and vk_now < 30,
        10,
        f"변동성 안정 (VKOSPI {vk_now:.1f})",
        f"변동성 과열 (VKOSPI {vk_now:.1f}) — 휩쏘·갭 리스크↑",
    )
else:
    reasons.append("⚠️ VKOSPI 데이터 미확보 — 변동성 항목 제외")

# 7. 최종 판정
pct = score / max_score * 100

print("─" * 45)
for r in reasons:
    print(r)
print("─" * 45)
print(f"\n📊 종가배팅 적합도 점수: {score} / {max_score} ({pct:.0f}%)\n")

if pct >= 70:
    print("🟢 [추천] 종가배팅에 우호적인 장세입니다. 스크리너를 돌려보세요.")
elif pct >= 45:
    print("🟡 [주의] 중립 구간입니다. 종목 수를 줄이고 손절선을 타이트하게 가져가세요.")
else:
    print("🔴 [비추천] 오늘은 종가배팅 환경이 좋지 않습니다.")
    print("    추세 이탈·거래 위축·고변동성 중 하나 이상 — 관망을 권합니다.")

🚦 [시장 레짐 필터] 오늘 종가배팅 적합도를 점검합니다...

─────────────────────────────────────────────
❌ 코스피 추세 약함 (지수<이평선) (0)
❌ 코스피 60일선 횡보/하락 (0)
❌ 코스닥 추세 약함 (지수<이평선) (0)
❌ 코스닥 60일선 횡보/하락 (0)
❌ 코스닥 거래량 위축 (최근5일 96%) (0)
⚠️ VKOSPI 데이터 미확보 — 변동성 항목 제외
─────────────────────────────────────────────

📊 종가배팅 적합도 점수: 0 / 90 (0%)

🔴 [비추천] 오늘은 종가배팅 환경이 좋지 않습니다.
    추세 이탈·거래 위축·고변동성 중 하나 이상 — 관망을 권합니다.


In [8]:
# 1. 필수 라이브러리 설치 및 불러오기 (코랩 최초 1회 실행)
!pip install -q finance-datareader tqdm

import datetime
import FinanceDataReader as fdr
import pandas as pd
from tqdm.notebook import tqdm

today_str = datetime.date.today().strftime("%Y-%m-%d")
print(f"🎯 === {today_str} 종가배팅 [바닥권 매물소화 대장주] 스크리닝 시작 ===")

# 2. 날짜 설정 (최근 6개월간의 매물대 및 바닥가 계산용)
start_date = (datetime.date.today() - datetime.timedelta(days=180)).strftime("%Y-%m-%d")

try:
    # 3. 국내 시장 전체 종목 리스트 획득
    df_krx = fdr.StockListing("KRX")
    df_krx = df_krx[df_krx["Market"].isin(["KOSPI", "KOSDAQ"])]

    # 데이터 정제 (숫자형 변환 및 결측치 제거)
    df_krx["Amount"] = pd.to_numeric(df_krx["Amount"], errors="coerce")
    df_krx["ChgRate"] = pd.to_numeric(df_krx["ChgRate"], errors="coerce")
    df_krx = df_krx.dropna(subset=["Amount", "ChgRate"])

    # 4. 1차 필터링: 오늘 거래대금 100억 이상 + 최소 3% 이상 상승 중인 힘 있는 주도주만 타겟
    df_filtered = df_krx[(df_krx["Amount"] >= 10000000000) & (df_krx["ChgRate"] >= 3.0)]

    print(f"▶ 1차 필터링 완료 (거래대금 100억 이상 & +3% 이상): {len(df_filtered)}개 종목 분석 중...")

    final_candidates = []

    # 5. 각 종목별 과거 데이터 분석 및 매물대 돌파 검증
    for idx, row in tqdm(df_filtered.iterrows(), total=len(df_filtered), desc="매물대 돌파 검증 중"):
        code = row["Code"]
        name = row["Name"]
        chg_rate = row["ChgRate"]
        amount_krw = row["Amount"]

        try:
            # 개별 종목 일봉 데이터 가져오기
            df_price = fdr.DataReader(code, start=start_date)
            if len(df_price) < 60:  # 신규 상장주 제외
                continue

            # 이동평균선 계산
            df_price["MA5"] = df_price["Close"].rolling(window=5).mean()
            df_price["MA20"] = df_price["Close"].rolling(window=20).mean()
            df_price["MA60"] = df_price["Close"].rolling(window=60).mean()

            # 최신 데이터 추출
            latest = df_price.iloc[-1]
            close_today = latest["Close"]
            vol_today = latest["Volume"]

            ma5_today = latest["MA5"]
            ma20_today = latest["MA20"]
            ma60_today = latest["MA60"]

            # 최근 5거래일 평균 거래량 (오늘 제외)
            avg_vol_5d = df_price["Volume"].iloc[-6:-1].mean() if len(df_price) >= 6 else 1

            # 바닥권 검증: 최근 120거래일(약 6개월) 최저가 대비 현재 주가 상승률 계산
            min_price_120d = df_price["Close"].tail(120).min()
            price_from_bottom = ((close_today / min_price_120d) - 1) * 100

            # ─── 🔍 반영된 핵심 조건 검증 ───
            # 조건 A: 거래량 폭발 (최근 5일 평균 거래량 대비 500% 이상 폭발)
            cond_vol = vol_today > (avg_vol_5d * 5.0)

            # 조건 B: 정배열 초기 및 매물대 상단 안착 (종가가 5일, 20일, 60일 이평선 모두를 뚫고 위에 위치)
            cond_ma = (close_today > ma5_today) and (close_today > ma20_today) and (close_today > ma60_today)

            # 조건 C: 완전 고점 추격매수 방지 (120일 최저가 대비 주가 상승률이 60% 이하인 '바닥권 탈출형'만)
            cond_bottom = price_from_bottom <= 60.0

            if cond_vol and cond_ma and cond_bottom:
                final_candidates.append({
                    "종목코드": code,
                    "종목명": name,
                    "현재가": f"{int(close_today):,}원",
                    "오늘등락률": f"{chg_rate:.2f}%",
                    "거래대금(억)": int(amount_krw / 100000000),
                    "바닥대비상승률": f"{price_from_bottom:.1f}%"
                })
        except:
            continue

    # 6. 결과 출력 (모바일 최적화 및 거래대금 순 정렬)
    print("\n🔥 [종가배팅 필터링 결과] 🔥")
    if final_candidates:
        df_result = pd.DataFrame(final_candidates)
        df_result = df_result.sort_values(by="거래대금(억)", ascending=False)
        display(df_result.style.hide(axis="index"))
    else:
        print("오늘 조건(거래대금 100억+ & 500% 거래폭발 & 정배열초기 & 바닥권 탈출)을 만족하는 종목이 없습니다.")
    print("\n" + "="*50)

except Exception as global_e:
    print(f"데이터 로드 실패: {global_e}")

🎯 === 2026-08-06 종가배팅 [바닥권 매물소화 대장주] 스크리닝 시작 ===
데이터 로드 실패: 'ChgRate'


In [9]:
# 1. 필요 라이브러리 설치 및 임포트
!pip install -q finance-datareader tqdm

import datetime
import FinanceDataReader as fdr
import pandas as pd
from tqdm.notebook import tqdm

print("🎯 [정통 스윙 타점] 주도주 60일선 기간조정 수렴 스크리너를 시작합니다...\n")

# 2. 분석 기간 설정 (60일선 및 기간조정 계산을 위해 150일 데이터 확보)
start_date = (datetime.date.today() - datetime.timedelta(days=160)).strftime(
    "%Y-%m-%d"
)

# 3. 국내 시장 전체 종목 리스트 획득
df_krx = fdr.StockListing("KRX")
df_krx = df_krx[df_krx["Market"].isin(["KOSPI", "KOSDAQ"])]

# 4. 최근 거래대금 상위 100개 종목 추출 (시장의 돈이 강하게 돌았던 주도주 후보군)
df_krx["Amount"] = pd.to_numeric(df_krx["Amount"], errors="coerce")
df_krx = df_krx.dropna(subset=["Amount"])
top_100 = df_krx.nlargest(100, "Amount").copy()

results = []

# 5. 영상 참조: 60일선 부근 기간조정 및 수급 손바뀜 검증
for idx, row in tqdm(
    top_100.iterrows(), total=len(top_100), desc="스윙 타점 계산 중"
):
    code = row["Code"]
    name = row["Name"]

    try:
        df_price = fdr.DataReader(code, start=start_date)
        if len(df_price) < 70:
            continue

        # 주요 이동평균선 계산
        df_price["MA5"] = df_price["Close"].rolling(window=5).mean()
        df_price["MA20"] = df_price["Close"].rolling(window=20).mean()
        df_price["MA60"] = df_price["Close"].rolling(window=60).mean()

        # 최신 데이터 추출
        latest = df_price.iloc[-1]
        close = latest["Close"]
        ma5 = latest["MA5"]
        ma20 = latest["MA20"]
        ma60 = latest["MA60"]

        # ─── 🔍 핵심 조건 1: 60일 이동평균선 '부근' 밀집 검증 ───
        # 주가가 대세 상승 후 60일선까지 얌전하게 내려와 지지받는 구간 (이탈 범위 ±4% 이내)
        if not (ma60 * 0.96 <= close <= ma60 * 1.04):
            continue

        # ─── 🔍 핵심 조건 2: 기간조정 (변동성 축소) 검증 ───
        # 최근 5거래일 동안 주가가 튀지 않고 60일선 주위에서 에너지를 웅축했는지 확인
        recent_5 = df_price.iloc[-5:]
        price_std = recent_5["Close"].std()  # 5일간 종가 표준편차
        mean_close = recent_5["Close"].mean()

        # 주가 움직임이 좁은 밴드(3% 이내)로 수렴하며 매물이 소화되는 '기간조정' 상태인지 판별
        if (price_std / mean_close) > 0.03:
            continue

        # ─── 🔍 핵심 조건 3: 대규모 손바뀜 흔적 (세력 개입 이력) ───
        # 최근 20거래일 이내에 평소 거래량 대비 4배 이상 대량 거래량이 터진 날이 있었는지 추적
        df_price["Vol_MA20_Prev"] = (
            df_price["Volume"].shift(1).rolling(window=20).mean()
        )
        recent_20 = df_price.iloc[-20:]
        has_handshake = any(
            recent_20["Volume"] > (recent_20["Vol_MA20_Prev"] * 4.0)
        )

        # 단위 변환 및 데이터 저장 (모바일 화면 폭에 맞춤)
        amount_idx = round(row["Amount"] / 100000000)  # 억 단위 변환
        results.append(
            {
                "종목명": name,
                "현재가": f"{int(close):,}",
                "60일선": f"{int(ma60):,}",
                "거래대금": f"{amount_idx:,}억",
                "손바뀜이력": "포착(🔥)" if has_handshake else "미포착",
            }
        )

    except:
        continue

# 6. 최종 스크리닝 결과 출력
print("\n📊 [영상 참조: 60일선 기간조정 수렴형 스윙 종목] 📊")
if results:
    final_df = pd.DataFrame(results)
    # 손바뀜이 일어난 핵심 종목이 상단에 보이도록 정렬
    final_df = final_df.sort_values(by="손바뀜이력", ascending=False)
    display(final_df.style.hide(axis="index"))
else:
    print(
        "오늘 60일선 부근에서 기간조정 수렴 및 거래대금 조건을 만족하는 종목이 없습니다."
    )

🎯 [정통 스윙 타점] 주도주 60일선 기간조정 수렴 스크리너를 시작합니다...



스윙 타점 계산 중:   0%|          | 0/100 [00:00<?, ?it/s]


📊 [영상 참조: 60일선 기간조정 수렴형 스윙 종목] 📊


종목명,현재가,60일선,거래대금,손바뀜이력
카카오,"38,300","38,106",895억,미포착
삼성화재,"648,000","626,016",713억,미포착
대한항공,"27,500","26,616",699억,미포착
KT&G,"182,700","179,078",527억,미포착
SK이노베이션,"108,600","110,655",525억,미포착


In [10]:
# 1. 필요 라이브러리 설치 및 임포트
!pip install -q finance-datareader tqdm

import datetime
import FinanceDataReader as fdr
import pandas as pd
from tqdm.notebook import tqdm

print("🎯 [순환매 길목지키기] 낙폭과대 실적·우량주 스크리닝을 시작합니다...\n")

# 2. 분석 기간 설정 (120일선 및 거래량 바닥 계산을 위해 200일 데이터 확보)
start_date = (datetime.date.today() - datetime.timedelta(days=250)).strftime(
    "%Y-%m-%d"
)

# 3. 국내 시장 전체 종목 리스트 획득
df_krx = fdr.StockListing("KRX")
# 시가총액이 너무 작은 잡주를 거르고 기관/외인의 수급이 들어올 수 있는 우량주 위주로 필터링
# 거래대금 상위 150개 종목을 후보군으로 선정
df_krx["Amount"] = pd.to_numeric(df_krx["Amount"], errors="coerce")
df_krx = df_krx.dropna(subset=["Amount"])
top_150 = df_krx.nlargest(150, "Amount").copy()

results = []

# 4. 순환매 길목(낙폭과대 + 거래량 마름) 조건 검증
for idx, row in tqdm(
    top_150.iterrows(), total=len(top_150), desc="바닥 다지기 종목 분석 중"
):
    code = row["Code"]
    name = row["Name"]

    try:
        df_price = fdr.DataReader(code, start=start_date)
        if len(df_price) < 130:
            continue

        # 주요 장기 이동평균선 계산
        df_price["MA60"] = df_price["Close"].rolling(window=60).mean()
        df_price["MA120"] = df_price["Close"].rolling(window=120).mean()

        # 최근 20거래일 평균 거래량 계산
        df_price["Vol_MA20"] = df_price["Volume"].rolling(window=20).mean()

        # 최신 데이터 추출
        latest = df_price.iloc[-1]
        close = latest["Close"]
        volume = latest["Volume"]
        ma60 = latest["MA60"]
        ma120 = latest["MA120"]
        vol_ma20 = latest["Vol_MA20"]

        # ─── 🔍 핵심 조건 1: 60일선 또는 120일선 장기 이평선 부근까지 낙폭과대 ───
        # 고점 대비 충분히 하락하여 장기 지지선(60일선 혹은 120일선)의 좁은 범위(±3%) 내에 안착했는지 확인
        is_near_ma60 = ma60 * 0.97 <= close <= ma60 * 1.03
        is_near_ma120 = ma120 * 0.97 <= close <= ma120 * 1.03

        if not (is_near_ma60 or is_near_ma120):
            continue

        # ─── 🔍 핵심 조건 2: 거래량 바짝 마름 (거래량 바닥 = 매도세 소멸) ───
        # 오늘 거래량이 최근 20일 평균 거래량의 60% 이하로 감소했는지 확인
        # 더 이상 팔 사람도 없고 사는 사람도 없어서 거래량이 완전히 죽은 '길목' 상태를 포착
        if volume > (vol_ma20 * 0.6):
            continue

        # ─── 🔍 핵심 조건 3: 주가 횡보 및 기간조정 상태 검증 ───
        # 최근 5거래일 동안 주가가 큰 변동 없이 바닥을 다지고 있는지 확인 (표준편차 2.5% 이내)
        recent_5 = df_price.iloc[-5:]
        if (recent_5["Close"].std() / recent_5["Close"].mean()) > 0.025:
            continue

        # 지지선 이름 판별
        support_line = "120일선" if is_near_ma120 else "60일선"

        # 데이터 저장 (모바일 맞춤형)
        amount_idx = round(row["Amount"] / 100000000)  # 억 단위
        results.append(
            {
                "종목명": name,
                "현재가": f"{int(close):,}",
                "지지선": support_line,
                "거래대금": f"{amount_idx:,}억",
                "거래량비율": f"{int((volume/vol_ma20)*100)}%",
            }
        )

    except:
        continue

# 5. 최종 결과 출력
print("\n📊 [순환매 선취매 타점: 장기 이평선 지지 + 거래량 바닥 종목] 📊")
if results:
    final_df = pd.DataFrame(results)
    # 거래량이 가장 많이 말라붙은(수치가 낮은) 순서대로 정렬하여 매수 최적 후보 제시
    final_df = final_df.sort_values(by="거래량비율", ascending=True)
    display(final_df.style.hide(axis="index"))
else:
    print(
        "오늘 장기 지지선 부근에서 거래량이 바짝 마른 낙폭과대 우량주가 없습니다."
    )


🎯 [순환매 길목지키기] 낙폭과대 실적·우량주 스크리닝을 시작합니다...



바닥 다지기 종목 분석 중:   0%|          | 0/150 [00:00<?, ?it/s]


📊 [순환매 선취매 타점: 장기 이평선 지지 + 거래량 바닥 종목] 📊


종목명,현재가,지지선,거래대금,거래량비율
SK이노베이션,"108,600",60일선,525억,52%


In [11]:
# 1. 라이브러리 설치 및 불러오기
!pip install -q --break-system-packages finance-datareader

import datetime
import FinanceDataReader as fdr
import pandas as pd

print("🌊 [스윙 레짐 모니터] 1주~1달 스윙 적합도를 점검합니다...\n")

# 2. 날짜 설정 (중기 추세 판단 위해 넉넉하게)
start_date = (datetime.date.today() - datetime.timedelta(days=300)).strftime("%Y-%m-%d")

# 3. 지수 데이터 로드
def load_index(symbol):
    df = fdr.DataReader(symbol, start=start_date)
    df["MA20"]  = df["Close"].rolling(20).mean()
    df["MA60"]  = df["Close"].rolling(60).mean()
    df["MA120"] = df["Close"].rolling(120).mean()
    df["Ret"]   = df["Close"].pct_change()
    return df

kospi  = load_index("KS11")
kosdaq = load_index("KQ11")

# ─── 채점 시스템 ───
score = 0
max_score = 0
reasons = []
warnings = []

def check(condition, weight, label_pass, label_fail, warn_if_fail=None):
    global score, max_score
    max_score += weight
    if condition:
        score += weight
        reasons.append(f"✅ {label_pass} (+{weight})")
    else:
        reasons.append(f"❌ {label_fail} (0)")
        if warn_if_fail:
            warnings.append(warn_if_fail)

# 4. 중기 추세 건강성 (스윙의 핵심 — 60일·120일선)
def swing_trend(df, name, w_mid, w_long, w_slope):
    latest = df.iloc[-1]
    close, ma60, ma120 = latest["Close"], latest["MA60"], latest["MA120"]
    ma60_10ago = df["MA60"].iloc[-11]

    check(
        close > ma60, w_mid,
        f"{name} 60일선 위 (중기 상승)",
        f"{name} 60일선 이탈 (중기 약세)",
        warn_if_fail=f"{name} 60일선 이탈 — 신규 스윙 진입 보류",
    )
    check(
        close > ma120, w_long,
        f"{name} 120일선 위 (대세 상승)",
        f"{name} 120일선 이탈 (대세 의심)",
    )
    check(
        ma60 > ma60_10ago, w_slope,
        f"{name} 60일선 우상향",
        f"{name} 60일선 꺾임",
        warn_if_fail=f"{name} 60일선 기울기 하락 전환 주의",
    )

swing_trend(kospi, "코스피", 15, 10, 10)
swing_trend(kosdaq, "코스닥", 15, 10, 10)

# 5. 정배열 점검 (60>120 = 중기 정배열)
for df, name in [(kospi, "코스피"), (kosdaq, "코스닥")]:
    latest = df.iloc[-1]
    check(
        latest["MA60"] > latest["MA120"], 5,
        f"{name} 중기 정배열(60>120)",
        f"{name} 역배열/혼조",
    )

# 6. 변동성 — VKOSPI 대신 코스피 일간수익률 표준편차(연율화)로 측정
#    (VKOSPI는 데이터 소스 차단으로 불안정 → 외부 의존성 제거)
recent_vol = kospi["Ret"].iloc[-20:].std() * (252 ** 0.5) * 100   # 최근 20일
base_vol   = kospi["Ret"].iloc[-60:-20].std() * (252 ** 0.5) * 100  # 직전 구간
check(
    recent_vol < base_vol * 1.3 and recent_vol < 30, 10,
    f"변동성 안정 (코스피 연율변동성 {recent_vol:.1f}%) — 추세 신뢰",
    f"변동성 상승 (코스피 연율변동성 {recent_vol:.1f}%) — 추세 흔들림 주의",
    warn_if_fail="변동성 급등 — 보유 비중 축소 검토",
)

# 7. 최종 판정
pct = score / max_score * 100

print("─" * 48)
for r in reasons:
    print(r)
print("─" * 48)

kp = kospi.iloc[-1]
print(f"\n📈 코스피 {kp['Close']:.0f} | 60일선 {kp['MA60']:.0f} | 120일선 {kp['MA120']:.0f}")
kq = kosdaq.iloc[-1]
print(f"📈 코스닥 {kq['Close']:.0f} | 60일선 {kq['MA60']:.0f} | 120일선 {kq['MA120']:.0f}")

print(f"\n📊 스윙 적합도 점수: {score} / {max_score} ({pct:.0f}%)\n")

if pct >= 70:
    print("🟢 [추천] 중기 추세 건강 — 신규 스윙 진입 우호적.")
    print("    1~4주 보유 전략 유효. 추세 추종 종목 위주로.")
elif pct >= 45:
    print("🟡 [선별 진입] 추세 혼조 — 강한 주도주만 선별, 비중 축소.")
    print("    손절선 -5~7% 타이트하게. 분할 진입 권장.")
else:
    print("🔴 [관망] 중기 추세 약화 — 신규 스윙 비추천.")
    print("    현금 비중 확대, 보유분 익절/손절 정리 검토.")

# 8. 이탈 경고 (배 타는 동안 이것만 봐도 됨)
if warnings:
    print("\n⚠️ ───── 보유자 경고 신호 ───── ⚠️")
    for w in warnings:
        print(f"  • {w}")
    print("  → 위 신호가 떴다면 보유 포지션 점검 필요")
else:
    print("\n✅ 주요 추세 이탈 신호 없음 — 보유 유지 가능 구간")

🌊 [스윙 레짐 모니터] 1주~1달 스윙 적합도를 점검합니다...

────────────────────────────────────────────────
❌ 코스피 60일선 이탈 (중기 약세) (0)
❌ 코스피 120일선 이탈 (대세 의심) (0)
❌ 코스피 60일선 꺾임 (0)
❌ 코스닥 60일선 이탈 (중기 약세) (0)
❌ 코스닥 120일선 이탈 (대세 의심) (0)
❌ 코스닥 60일선 꺾임 (0)
✅ 코스피 중기 정배열(60>120) (+5)
❌ 코스닥 역배열/혼조 (0)
❌ 변동성 상승 (코스피 연율변동성 102.2%) — 추세 흔들림 주의 (0)
────────────────────────────────────────────────

📈 코스피 6296 | 60일선 7649 | 120일선 6786
📈 코스닥 802 | 60일선 924 | 120일선 1033

📊 스윙 적합도 점수: 5 / 90 (6%)

🔴 [관망] 중기 추세 약화 — 신규 스윙 비추천.
    현금 비중 확대, 보유분 익절/손절 정리 검토.

⚠️ ───── 보유자 경고 신호 ───── ⚠️
  • 코스피 60일선 이탈 — 신규 스윙 진입 보류
  • 코스피 60일선 기울기 하락 전환 주의
  • 코스닥 60일선 이탈 — 신규 스윙 진입 보류
  • 코스닥 60일선 기울기 하락 전환 주의
  • 변동성 급등 — 보유 비중 축소 검토
  → 위 신호가 떴다면 보유 포지션 점검 필요
